In [ ]:
# Lung Cancer Prediction using Sagemaker Training Pipeline
# Used the prepared file from S3 featues/ folder -> data_preparation_handoff.ipynb

# Imports & Setup

In [19]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import os
import io
import time
from sagemaker.inputs import TrainingInput
from sagemaker import image_uris, get_execution_role
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, confusion_matrix
)
 
session   = sagemaker.Session()
role      = get_execution_role()
bucket    = "prognostica-cancer-project"
prefix    = "lung-cancer"
s3_client = boto3.client("s3")
 
print(f"Role   : {role}")
print(f"Bucket : {bucket}")
print(f"Region : {session.boto_region_name}")

Role   : arn:aws:iam::381491860224:role/LabRole
Bucket : prognostica-cancer-project
Region : us-east-1


#  Load and encode files from data preparation

In [20]:
os.makedirs("data", exist_ok=True)

def load_and_encode(s3_key, name):
    obj = s3_client.get_object(Bucket=bucket, Key=s3_key)
    df  = pd.read_csv(io.BytesIO(obj["Body"].read()))
    print(f"\n{name} raw shape: {df.shape}")

    # Encode target
    df["lung_cancer"] = df["lung_cancer"].map({"YES": 1, "NO": 0})

    # Encode gender
    df["gender"] = df["gender"].map({"M": 1, "F": 0})

    # Convert boolean age group columns to int
    bool_cols = df.select_dtypes(include="bool").columns.tolist()
    df[bool_cols] = df[bool_cols].astype(int)

    # Verify no remaining string columns
    bad = df.select_dtypes(include="object").columns.tolist()
    if bad:
        raise ValueError(f"Non-numeric columns remaining: {bad}")

    # Move target to first column (SageMaker XGBoost requirement)
    cols = ["lung_cancer"] + [c for c in df.columns if c != "lung_cancer"]
    df   = df[cols]

    print(f"{name} encoded shape: {df.shape}")
    print(f"Target counts: {df['lung_cancer'].value_counts().to_dict()}")
    print(f"Sample:\n{df.head(2)}")
    return df

train_df = load_and_encode("features/lung_train.csv", "TRAIN")
val_df   = load_and_encode("features/lung_val.csv",   "VAL")
test_df  = load_and_encode("features/lung_test.csv",  "TEST")


TRAIN raw shape: (13699, 20)
TRAIN encoded shape: (13699, 20)
Target counts: {1: 11908, 0: 1791}
Sample:
   lung_cancer  gender  age  smoking  yellow_fingers  anxiety  peer_pressure  \
0            1       1   60        1               2        1              2   
1            1       0   55        2               1        2              2   

   chronic_disease  fatigue  allergy  wheezing  alcohol_consuming  coughing  \
0                1        2        2         1                  1         2   
1                2        1        2         2                  1         1   

   shortness_of_breath  swallowing_difficulty  chest_pain  age_group_41-50  \
0                    1                      1           2                0   
1                    2                      1           2                0   

   age_group_51-60  age_group_61-70  age_group_71+  
0                1                0              0  
1                1                0              0  

VAL raw shape: (2936

# Upload encoded files to S3 for SageMaker

In [21]:
def upload_encoded(df, name, s3_key):
    local = f"data/lung_{name}_ready.csv"
    df.to_csv(local, index=False, header=False)
    s3_client.upload_file(local, bucket, s3_key)
    print(f"Uploaded {name} -> s3://{bucket}/{s3_key}  ({df.shape})")

upload_encoded(train_df, "train", f"{prefix}/train/lung_train.csv")
upload_encoded(val_df,   "val",   f"{prefix}/val/lung_val.csv")
upload_encoded(test_df,  "test",  f"{prefix}/test/lung_test.csv")

train_input = TrainingInput(
    f"s3://{bucket}/{prefix}/train/", content_type="text/csv"
)
print("\nData ready for training.")

Uploaded train -> s3://prognostica-cancer-project/lung-cancer/train/lung_train.csv  ((13699, 20))
Uploaded val -> s3://prognostica-cancer-project/lung-cancer/val/lung_val.csv  ((2936, 20))
Uploaded test -> s3://prognostica-cancer-project/lung-cancer/test/lung_test.csv  ((2936, 20))

Data ready for training.


# Define XGBoost Estimator


In [22]:
yes_count        = (train_df["lung_cancer"] == 1).sum()
no_count         = (train_df["lung_cancer"] == 0).sum()
scale_pos_weight = round(yes_count / no_count, 4)
print(f"scale_pos_weight = {scale_pos_weight}  (YES:{yes_count} / NO:{no_count})")

container = image_uris.retrieve(
    framework="xgboost",
    region=session.boto_region_name,
    version="1.5-1",
)
print(f"Container: {container}")

xgb = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session,
    hyperparameters={
        "objective":        "binary:logistic",
        "num_round":        100,
        "max_depth":        4,
        "eta":              0.1,
        "subsample":        0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": scale_pos_weight,
    },
)
print("Estimator ready.")

scale_pos_weight = 6.6488  (YES:11908 / NO:1791)


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


Container: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.5-1
Estimator ready.


# Train

In [23]:
xgb.fit(
    inputs={"train": train_input},
    logs=True,
    wait=True,
)

print("\nTraining complete!")
print(f"Model artifact -> {xgb.model_data}")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-24-02-01-02-544


2026-03-24 02:01:02 Starting - Starting the training job...
2026-03-24 02:01:35 Starting - Preparing the instances for training...
2026-03-24 02:01:57 Downloading - Downloading input data...
2026-03-24 02:02:23 Downloading - Downloading the training image...
2026-03-24 02:03:03 Training - Training image download completed. Training in progress.../miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-03-24 02:03:15.524 ip-10-0-184-92.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-24 02:03:15.546 ip-10-0-184-92.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-24:02:03:15:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-24:02:03:15:INFO] Failed to parse hyperparameter objective value binary:l

# Batch Transform 

In [24]:
# Upload test features only (no target column)
test_features = test_df.iloc[:, 1:]
test_features.to_csv("data/lung_test_features.csv", index=False, header=False)
s3_client.upload_file(
    "data/lung_test_features.csv", bucket,
    f"{prefix}/test_features/lung_test_features.csv"
)
print(f"Test features: {test_features.shape}")

# Unique output path per run - avoids stale cached predictions
unique_output = f"s3://{bucket}/{prefix}/predictions/{int(time.time())}"

transformer = xgb.transformer(
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=unique_output,
    assemble_with="Line",
    accept="text/csv",
)

transformer.transform(
    data=f"s3://{bucket}/{prefix}/test_features/",
    content_type="text/csv",
    split_type="Line",
    wait=True,
    logs=True,
)

print(f"\nBatch transform complete!")
print(f"Predictions -> {unique_output}")

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-03-24-02-03-49-846


Test features: (2936, 19)


INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-03-24-02-03-50-556


................................/miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-03-24:02:09:14:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-24:02:09:14:INFO] No GPUs detected (normal if no gpus installed)
[2026-03-24:02:09:14:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_ad

# Evaluate 


In [25]:
s3_client.download_file(
    bucket,
    unique_output.replace(f"s3://{bucket}/", "") + "/lung_test_features.csv.out",
    "data/lung_predictions.csv"
)

probs  = pd.read_csv("data/lung_predictions.csv", header=None)[0]
labels = test_df["lung_cancer"].reset_index(drop=True)

print(f"\nPrediction stats:")
print(probs.describe())
print(f"\nUnique prediction values: {len(probs.unique())}")

# Evaluate at default threshold
preds = (probs >= 0.5).astype(int)

print("\n" + "=" * 55)
print("  EVALUATION RESULTS")
print("=" * 55)
print(f"  Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(labels, probs):.4f}")

print("\nConfusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(labels, preds),
    index=["Actual NO", "Actual YES"],
    columns=["Pred NO", "Pred YES"]
))

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["NO (0)", "YES (1)"]))

print("\n" + "=" * 55)
print("  TRAINING JOB SUMMARY")
print("=" * 55)
print(f"  Job name      : {xgb.latest_training_job.name}")
print(f"  Model artifact: {xgb.model_data}")
print(f"  Instance type : ml.m5.xlarge")
print(f"  Framework     : XGBoost 1.5-1")
print(f"  Train rows    : {len(train_df)}")
print(f"  Test rows     : {len(test_df)}")
print(f"  Features      : {test_features.shape[1]}")


Prediction stats:
count    2936.000000
mean        0.977519
std         0.005257
min         0.913927
25%         0.975069
50%         0.977805
75%         0.980360
max         0.995523
Name: 0, dtype: float64

Unique prediction values: 2900

  EVALUATION RESULTS
  Accuracy : 0.8692
  ROC-AUC  : 0.4986

Confusion Matrix:
            Pred NO  Pred YES
Actual NO         0       384
Actual YES        0      2552

Classification Report:
              precision    recall  f1-score   support

      NO (0)       0.00      0.00      0.00       384
     YES (1)       0.87      1.00      0.93      2552

    accuracy                           0.87      2936
   macro avg       0.43      0.50      0.47      2936
weighted avg       0.76      0.87      0.81      2936


  TRAINING JOB SUMMARY
  Job name      : sagemaker-xgboost-2026-03-24-02-01-02-544
  Model artifact: s3://prognostica-cancer-project/lung-cancer/output/sagemaker-xgboost-2026-03-24-02-01-02-544/output/model.tar.gz
  Instance type : ml

/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [26]:
from sagemaker import TrainingJobAnalytics

metrics = TrainingJobAnalytics(
    training_job_name=xgb.latest_training_job.name
).dataframe()

print(f"Total rounds trained: {len(metrics)}")
print(metrics)

Total rounds trained: 1
   timestamp    metric_name   value
0        0.0  train:logloss  0.5018
